# 01: Data collection (Phase 1)

**Goal:** load the SEC Financial Statement Data Sets, select the annual 10-K filings of SIC 3674 (semiconductor) companies, verify the four case companies by CIK, count companies per year, and shrink the huge `num` files to just the selected filings.

**Inputs:** quarterly zips in `data/raw` (2014q3 to 2026q3), read in place, never modified.

**Outputs:**
- `data/interim/filings_3674.csv`: one row per 10-K / 10-K/A filing of a SIC 3674 company
- `data/processed/counts_by_label_year.csv`: companies and filings per label year
- `data/interim/num_3674.parquet`: every `num` row belonging to those filings

Field names were checked against `readme.htm` inside the zips.

In [1]:
import sys
import pandas as pd

# Make the project root importable so `src` works when the notebook runs from notebooks/.
sys.path.insert(0, "..")
from src import config
from src.load_sec import list_quarter_zips, check_layouts, load_all_sub, load_num_for_filings
from src.preprocess import label_year_from_period

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 100)
print("Python", sys.version.split()[0], "| pandas", pd.__version__)

Python 3.12.4 | pandas 3.1.0


## Step 1: List the quarterly files and check they share one layout

Before stacking quarters, every `sub.txt` and `num.txt` must have exactly the columns the SEC readme defines; otherwise rows from different quarters would be misaligned.

In [2]:
zips = list_quarter_zips(config.RAW_DIR)
print(f"{len(zips)} quarterly zips: {zips[0].stem} to {zips[-1].stem}")

layout = check_layouts(zips)
print("Quarters with the expected sub columns:", layout["sub_columns_ok"].sum(), "of", len(layout))
print("Quarters with the expected num columns:", layout["num_columns_ok"].sum(), "of", len(layout))
layout[~(layout["sub_columns_ok"] & layout["num_columns_ok"])]  # empty table = no layout problems

49 quarterly zips: 2014q3 to 2026q3
Quarters with the expected sub columns: 49 of 49
Quarters with the expected num columns: 49 of 49


,quarter,sub_columns_ok,num_columns_ok


## Step 2: Stack the submission (`sub`) files

One row per filing, with a `source_quarter` column recording which zip it came from. The parse check compares rows read by pandas with the raw line count of each file; they must be equal.

In [3]:
sub, sub_check = load_all_sub(zips)
print(f"Stacked sub table: {len(sub):,} filings from {sub['source_quarter'].nunique()} quarters")
print("Quarters where rows parsed != lines in file:", (sub_check["rows_parsed"] != sub_check["lines_in_file"]).sum())
sub_check.head()

Stacked sub table: 337,436 filings from 49 quarters
Quarters where rows parsed != lines in file: 0


,quarter,rows_parsed,lines_in_file
0,2014q3,7790,7790
1,2014q4,7713,7713
2,2015q1,6995,6995
3,2015q2,8212,8212
4,2015q3,7455,7455


## Step 3: Filter to annual filings of SIC 3674 companies

SIC 3674 = *Semiconductors and Related Devices*. The readme notes `sic` is recorded as of each filing date. First, which forms do SIC 3674 companies file?

In [4]:
sic_3674 = sub[sub["sic"] == config.SIC_CODE]
sic_3674["form"].value_counts().rename("filings").to_frame().head(15)

,filings
form,
10-Q,3376
10-K,1096
20-F,266
6-K,87
S-1/A,59
S-1,54
10-Q/A,42
POS AM,34
10-K/A,32


We keep `10-K` (annual report) and `10-K/A` (its amendment; kept only so Phase 2 can deduplicate). Excluded on purpose:
- `10-KT`: transition reports for a shortened period after a fiscal year-end change, not a full year;
- `20-F` / `40-F`: annual reports of foreign private issuers, which follow different rules and often IFRS, so they are not comparable with 10-K filers;
- quarterly (`10-Q`) and other forms.

`label_year` is the Compustat-style year (fiscal year-end in January to May gives the previous calendar year). Phase 2 formalises it; here it is used to count filings per year.

In [5]:
filings = sic_3674[sic_3674["form"].isin(config.ANNUAL_FORMS)].copy()
filings["label_year"] = label_year_from_period(filings["period"])
filings = filings.sort_values(["cik", "period", "filed"]).reset_index(drop=True)

print(f"Annual filings (10-K and 10-K/A) of SIC {config.SIC_CODE}: {len(filings):,}")
print(filings["form"].value_counts().to_string())
print(f"Distinct companies (CIK): {filings['cik'].nunique()}")
print("Excluded 10-KT filings:", (sic_3674["form"] == "10-KT").sum())

filings.to_csv(config.INTERIM_DIR / "filings_3674.csv", index=False)
print("Saved", config.INTERIM_DIR / "filings_3674.csv")

Annual filings (10-K and 10-K/A) of SIC 3674: 1,128
form
10-K      1096
10-K/A      32
Distinct companies (CIK): 167
Excluded 10-KT filings: 4
Saved G:\DJSCE\Fourth Year\FADS\Mini Proj\data\interim\filings_3674.csv


## Step 4: Verify the four case companies by CIK

Companies are identified by CIK, never by name: Wolfspeed filed as *Cree* before late 2021. Here the CIKs from `config.py` are looked up in the full `sub` table, without the SIC filter, to confirm who each CIK is, whether its SIC is 3674 in every filing, and that every year is present.

In [6]:
case_all = sub[sub["cik"].isin(config.CASE_COMPANIES) & sub["form"].isin(config.ANNUAL_FORMS)].copy()
case_all["company"] = case_all["cik"].map(config.CASE_COMPANIES)
case_all["label_year"] = label_year_from_period(case_all["period"])

# Identity check: every name and former name the SEC recorded for each CIK,
# the SIC codes and fiscal year-ends seen, and the number of annual filings.
identity = case_all.groupby(["company", "cik"]).agg(
    names_recorded=("name", lambda s: " | ".join(dict.fromkeys(s))),
    former_names=("former", lambda s: " | ".join(dict.fromkeys(s.dropna()))),
    sic_codes=("sic", lambda s: sorted(s.dropna().unique().tolist())),
    fiscal_year_end_mmdd=("fye", lambda s: sorted(s.dropna().unique().tolist())),
    annual_filings=("adsh", "count"),
)
identity

,,names_recorded,former_names,sic_codes,fiscal_year_end_mmdd,annual_filings
company,cik,,,,,
AMD,2488,ADVANCED MICRO DEVICES INC,,[3674],[1231],12
Intel,50863,INTEL CORP,,[3674],[1231],12
Nvidia,1045810,NVIDIA CORP,NVIDIA CORP/DE,[3674],[0131],12
Wolfspeed,895419,"CREE INC | CREE, INC. | WOLFSPEED, INC.","CREE RESEARCH INC /NC/ | CREE INC | CREE, INC.",[3674],[0630],13


In [7]:
# Filings per company per label year. A 1 everywhere means one 10-K per year;
# a 2 would mean an amendment or a duplicate to resolve in Phase 2.
case_grid = case_all.pivot_table(index="label_year", columns="company", values="adsh", aggfunc="count", fill_value=0)
case_grid

company,AMD,Intel,Nvidia,Wolfspeed
label_year,,,,
2014,1,1,1,1
2015,1,1,1,1
2016,1,1,1,1
2017,1,1,1,1
2018,1,1,1,1
2019,1,1,1,1
2020,1,1,1,1
2021,1,1,1,1
2022,1,1,1,1


In [8]:
# Detail for each case filing: fiscal year-end (period), SEC fiscal year (fy), filing date and source quarter.
case_all.sort_values(["company", "period"])[
    ["company", "cik", "name", "form", "period", "fy", "label_year", "filed", "prevrpt", "source_quarter"]
].reset_index(drop=True)

,company,cik,name,form,period,fy,label_year,filed,prevrpt,source_quarter
0,AMD,2488,ADVANCED MICRO DEVICES INC,10-K,2014-12-31,2014,2014,2015-02-19,0,2015q1
1,AMD,2488,ADVANCED MICRO DEVICES INC,10-K,2015-12-31,2015,2015,2016-02-18,0,2016q1
2,AMD,2488,ADVANCED MICRO DEVICES INC,10-K,2016-12-31,2016,2016,2017-02-21,0,2017q1
3,AMD,2488,ADVANCED MICRO DEVICES INC,10-K,2017-12-31,2017,2017,2018-02-27,0,2018q1
4,AMD,2488,ADVANCED MICRO DEVICES INC,10-K,2018-12-31,2018,2018,2019-02-08,0,2019q1
5,AMD,2488,ADVANCED MICRO DEVICES INC,10-K,2019-12-31,2019,2019,2020-02-04,0,2020q1
6,AMD,2488,ADVANCED MICRO DEVICES INC,10-K,2020-12-31,2020,2020,2021-01-29,0,2021q1
7,AMD,2488,ADVANCED MICRO DEVICES INC,10-K,2021-12-31,2021,2021,2022-02-03,0,2022q1
8,AMD,2488,ADVANCED MICRO DEVICES INC,10-K,2022-12-31,2022,2022,2023-02-27,0,2023q1
9,AMD,2488,ADVANCED MICRO DEVICES INC,10-K,2023-12-31,2023,2023,2024-01-31,0,2024q1


In [9]:
# Missing-year check over the study range, including the year before the start
# (needed as the base year for 2015 growth ratios).
years_needed = range(config.LABEL_YEAR_START - 1, config.LABEL_YEAR_END + 1)
for company in config.CASE_COMPANIES.values():
    rows = case_all[case_all["company"] == company]
    missing = [y for y in years_needed if y not in set(rows["label_year"])]
    in_3674 = rows["sic"].eq(config.SIC_CODE).all()
    print(f"{company:10s} missing label years {years_needed.start}-{years_needed.stop - 1}: {missing or 'none'}"
          f" | SIC 3674 in every filing: {in_3674}")

Nvidia     missing label years 2014-2025: none | SIC 3674 in every filing: True
AMD        missing label years 2014-2025: none | SIC 3674 in every filing: True
Intel      missing label years 2014-2025: none | SIC 3674 in every filing: True
Wolfspeed  missing label years 2014-2025: none | SIC 3674 in every filing: True


## Step 5: Companies and filings per label year

The downloaded quarters (filed 2014 Q3 to 2026 Q3) fully cover label years 2014 to 2025. Label year 2013 only contains late fiscal years (e.g. May 2014 year-ends filed after July 2014), and label year 2026 only June to August 2026 year-ends filed by September 2026, so both edge years are partial and are marked as such.

In [10]:
counts = filings.groupby("label_year").agg(
    companies=("cik", "nunique"),
    filings=("adsh", "count"),
    form_10K=("form", lambda s: (s == "10-K").sum()),
    form_10KA=("form", lambda s: (s == "10-K/A").sum()),
)
# Companies with more than one annual filing in the same label year
# (amendments or duplicates; resolved in Phase 2).
per_company_year = filings.groupby(["label_year", "cik"]).size()
counts["companies_with_multiple_filings"] = (per_company_year > 1).groupby("label_year").sum()

complete = range(config.LABEL_YEAR_START - 1, config.LABEL_YEAR_END + 1)
counts["coverage"] = ["complete" if y in complete else "partial (edge of download)" for y in counts.index]
counts["below_30_companies"] = counts["companies"] < 30

counts.to_csv(config.PROCESSED_DIR / "counts_by_label_year.csv")
print("Saved", config.PROCESSED_DIR / "counts_by_label_year.csv")
counts

Saved G:\DJSCE\Fourth Year\FADS\Mini Proj\data\processed\counts_by_label_year.csv


,companies,filings,form_10K,form_10KA,companies_with_multiple_filings,coverage,below_30_companies
label_year,,,,,,,
2011,1,1,1,0,0,partial (edge of download),True
2012,1,1,1,0,0,partial (edge of download),True
2013,4,4,2,2,0,partial (edge of download),True
2014,114,116,114,2,2,complete,False
2015,100,102,100,2,2,complete,False
2016,90,93,88,5,3,complete,False
2017,91,92,91,1,1,complete,False
2018,89,91,89,2,2,complete,False
2019,87,90,87,3,3,complete,False


## Step 6: Keep only the `num` rows of the selected filings

Each `num.txt` is about 500 MB, so it is read in chunks and each chunk is filtered to the selected filings (`adsh`) before the next is read. No value-level filtering yet (date, unit, `coreg`, `segments`); that is Phase 2. This cell takes several minutes.

In [11]:
# Selected filings: the SIC 3674 annual filings, plus any case-company annual
# filing (case companies are kept by CIK even if their SIC were ever different).
adsh_keep = set(filings["adsh"]) | set(case_all["adsh"])
print(f"Filings to keep num rows for: {len(adsh_keep):,}")

num, num_check = load_num_for_filings(zips, adsh_keep)

Filings to keep num rows for: 1,128


2014q3: parsed 2,936,590 rows, kept 5,717


2014q4: parsed 2,940,236 rows, kept 7,335


2015q1: parsed 3,322,479 rows, kept 32,866


2015q2: parsed 2,588,598 rows, kept 8,107


...

2025q3: parsed 3,720,081 rows, kept 3,734


2025q4: parsed 3,832,977 rows, kept 4,583


2026q1: parsed 3,690,955 rows, kept 22,379


2026q2: parsed 3,608,711 rows, kept 4,139


2026q3: parsed 3,905,397 rows, kept 3,878


In [12]:
print("Quarters where rows parsed != lines in file:", (num_check["rows_parsed"] != num_check["lines_in_file"]).sum())
print(f"num rows read across all quarters: {num_check['rows_parsed'].sum():,}")
print(f"num rows kept:                     {len(num):,} ({len(num) / num_check['rows_parsed'].sum():.2%})")

# Every selected filing should have numbers in num; list any that do not.
no_numbers = sorted(adsh_keep - set(num["adsh"]))
print(f"Selected filings with no num rows: {len(no_numbers)}")
filings[filings["adsh"].isin(no_numbers)][["adsh", "cik", "name", "form", "period", "label_year", "source_quarter"]]

Quarters where rows parsed != lines in file: 0
num rows read across all quarters: 152,323,720
num rows kept:                     491,614 (0.32%)


Selected filings with no num rows: 0


,adsh,cik,name,form,period,label_year,source_quarter


In [13]:
# Do a filing's numbers ever appear in a different quarter's num file? (Expected: no.)
quarter_of_filing = pd.concat([filings, case_all])[["adsh", "source_quarter"]].drop_duplicates().set_index("adsh")["source_quarter"]
mismatch = (num["source_quarter"] != num["adsh"].map(quarter_of_filing)).sum()
print("num rows whose quarter differs from their filing's quarter:", mismatch)

num.to_parquet(config.INTERIM_DIR / "num_3674.parquet", index=False)
print("Saved", config.INTERIM_DIR / "num_3674.parquet")
num.head()

num rows whose quarter differs from their filing's quarter: 0


Saved G:\DJSCE\Fourth Year\FADS\Mini Proj\data\interim\num_3674.parquet


,adsh,tag,version,ddate,qtrs,uom,segments,coreg,value,source_quarter
0,0000316793-14-000021,OtherComprehensiveIncomeLossNetOfTaxPortionAtt...,us-gaap/2013,2013-06-30,4,USD,EquityComponents=AccumulatedOtherComprehensive...,NaN,-2549000.0,2014q3
1,0000791907-14-000099,OperatingIncomeLoss,us-gaap/2013,2012-06-30,4,USD,NaN,NaN,582036000.0,2014q3
2,0001445305-14-003896,ConvertibleLongTermNotesPayable,us-gaap/2013,2014-06-30,0,USD,NaN,NaN,0.0,2014q3
3,0001047469-14-007228,InterestPaidNet,us-gaap/2013,2014-06-30,4,USD,NaN,NaN,4659000.0,2014q3
4,0001047469-14-007228,OperatingIncomeLoss,us-gaap/2013,2014-06-30,4,USD,NaN,NaN,81295000.0,2014q3


In [14]:
# A first look at what the kept rows contain (informs Phase 2; nothing is dropped here).
print("Units (uom), top 5:")
print(num["uom"].value_counts().head(5).to_string())
print()
print("Share of rows with a coreg value:   ", f"{num['coreg'].notna().mean():.2%}")
print("Share of rows with a segments value:", f"{num['segments'].notna().mean():.2%}")
print()
print("qtrs values, top 5:")
print(num["qtrs"].value_counts().head(5).to_string())

Units (uom), top 5:
uom
USD       448804
shares     41213
pure         967
EUR          484
CNY           32

Share of rows with a coreg value:    0.94%
Share of rows with a segments value: 35.37%

qtrs values, top 5:
qtrs
4    306605
0    151279
1     31810
2       875
3       831


## Checks before moving on (from the project guide)

- Does each case company appear in every year? See Step 4.
- Roughly how many companies per year? See Step 5 (`below_30_companies`).
- Are the quarterly files consistent? See Step 1 (layout) and the parse checks in Steps 2 and 6.

Observations from these outputs are recorded in `docs/data_log.md`.